In [75]:
using NCDatasets
using CairoMakie
using GeoMakie
using Dates
using JupyterFormatter
enable_autoformat()

[ Info: Precompiling JupyterFormatter [b8b539d8-55b4-4e60-a505-d7876c054e58] (cache misses: wrong dep version loaded (10), incompatible header (10))

SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up
[ Info: Precompiling FilePathsGlobExt [db3c847f-822f-5018-835c-4bd87b334709] (cache misses: wrong dep version loaded (2), incompatible header (2))

SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up


1-element Vector{Function}:
 format_current_cell (generic function with 1 method)

In [ ]:
"""
    compute_psi_vorticity(resfile, grdfile)

Compute the normalised relative vorticity ζ = (∂v/∂x - ∂u/∂y) /f.
"""
function compute_surf_vorticity(resfile::String, grdfile::String)

    NCDataset(resfile) do ds_hist
        NCDataset(grdfile) do ds_grid

            # Load time vector
            thedates = ds_hist["time"][:]

            # Read 2D metrics correctly (keep as matrices)
            pm = ds_grid["pm"][:, :]   # 2D, rho-points
            pn = ds_grid["pn"][:, :]   # 2D, rho-points

            lon_rho = ds_grid["lon_rho"][:, :]
            lat_rho = ds_grid["lat_rho"][:, :]

            f = ds_grid["f"][:, :]

            u = ds_hist["u"][:, :, end, :]   # (xi_u, eta_u, s_rho) = (Lm, Mm+1, N)
            v = ds_hist["v"][:, :, end, :]   # (xi_v, eta_v, s_rho) = (Lm+1, Mm, N)

            Lm = size(u, 1)          # number of psi points in xi direction
            Mm = size(v, 2)          # number of psi points in eta direction
            ntimes = size(u, 3)

            zeta = zeros(eltype(u), Lm, Mm, ntimes)

            for k = 1:ntimes
                # ∂v/∂x at psi-points (difference in xi direction)
                # v[1:Lm+1, 1:Mm, k]  →  shape (Lm+1, Mm)
                pm_avg_x = @views (pm[1:Lm, 1:Mm] + pm[2:Lm+1, 1:Mm]) / 2   # (Lm, Mm)
                dvdx = @views (v[2:Lm+1, 1:Mm, k] - v[1:Lm, 1:Mm, k]) .* pm_avg_x

                # ∂u/∂y at psi-points (difference in eta direction)
                # u[1:Lm, 1:Mm+1, k]  →  shape (Lm, Mm+1)
                pn_avg_y = @views (pn[1:Lm, 1:Mm] + pn[1:Lm, 2:Mm+1]) / 2   # (Lm, Mm)
                dudy = @views (u[1:Lm, 2:Mm+1, k] - u[1:Lm, 1:Mm, k]) .* pn_avg_y

                # Both dvdx and dudy are now (Lm, Mm) → subtract
                @views zeta[:, :, k] .= dvdx .- dudy ./ f[1:end-1, 1:end-1]
            end
            replace!(zeta, 0.0 => NaN)
            return zeta::Array{Float32,3},
            lon_rho::Matrix{Float64},
            lat_rho::Matrix{Float64},
            thedates::Vector{DateTime}

        end
    end


end

In [ ]:
if gethostname() == "ctroupin-laptop"
    dirname = "My Passport"
elseif gethostname() == "ctroupin-PRIME-A320M-C-R2-0"
    dirname = "T7Shield"
else
    @error("Unknown host name")
end

datadir = "/media/ctroupin/$(dirname)/CROCO_FILES/run_nea_hermione"
figdir = "../figures/"
grdfile1 = joinpath(datadir, "croco_grd_nea.nc")
grdfile2 = joinpath(datadir, "croco_grd_nea.nc.1")
avgfile1 = joinpath(datadir, "croco_canary_avg.00110.nc")
avgfile2 = joinpath(datadir, "croco_canary_avg.00110.nc.1")

In [78]:
zeta, lon_zeta, lat_zeta, thedates = compute_surf_vorticity(avgfile2, grdfile2);

In [81]:
for (ii, tt) in enumerate(thedates)
    @info("Working on $(tt)")

    fig = Figure(size = (800, 800))
    ax = GeoAxis(
        fig[1, 1],
        dest = "+proj=merc",
        title = titlecase("relative vorticity\n $(tt)"),
        titlesize = 18,
        xgridcolor = :lightgray,
        xgridwidth = 0.5,
        xgridstyle = :dash,
        ygridcolor = :lightgray,
        ygridwidth = 0.5,
        ygridstyle = :dash,
    )#, )

    xlims!(ax, -20.0, -9.0)
    ylims!(ax, 24.0, 33.0)
    ax.xticks = collect(-50.0:2:0.0)
    ax.yticks = collect(20.0:2.0:45.0)

    hm = surface!(
        ax,
        lon_zeta,
        lat_zeta,
        zeta[:, :, ii],
        colormap = :curl,
        shading = NoShading,
        nan_color = :gray,
        interpolate = false,
        colorrange = (-1.5, 1.5),
    )
    lines!(ax, GeoMakie.coastlines(10), color = :black, linewidth = 0.7)

    Colorbar(
        fig[1, 2],
        hm,
        label = "ζ/f",
        labelrotation = 0,
        height = @lift($(pixelarea(ax.scene)).widths[2])
    )
    datestring = Dates.format(tt, "yyyymmdd_HHMMSS")
    figname = joinpath(figdir, "vorticity_$(datestring).png")
    save(figname, fig)
    # display(fig)
end

[ Info: Working on 2022-09-28T15:01:20
[ Info: Working on 2022-09-28T21:01:20
[ Info: Working on 2022-09-29T03:01:20
[ Info: Working on 2022-09-29T09:01:20
[ Info: Working on 2022-09-29T15:01:20
